In [36]:
# Imports

import os
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

In [37]:
# project paths
PROJECT_ROOT = Path("..").resolve()
MODELS_DIR = PROJECT_ROOT / "models"

HGB_MODEL_PATH = MODELS_DIR / "hgb_model.joblib"

hgb_model = joblib.load(HGB_MODEL_PATH)
print("HGB model loaded successfully")

HGB model loaded successfully


In [38]:
# feature names from fitted model

feature_cols = list(hgb_model.feature_names_in_)

print("Recovered feature names from trained HGB model")
print("Feature count:", len(feature_cols))
print(feature_cols)

Recovered feature names from trained HGB model
Feature count: 31
['temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa', 'precipitation_mm', 'biomass', 'fossil_gas', 'fossil_hard_coal', 'hydro_pumped_storage', 'hydro_run_of_river_and_poundage', 'nuclear', 'other', 'solar', 'wind_offshore', 'wind_onshore', 'totaloutput_mw', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'carbon_lag_48', 'carbon_lag_336', 'carbon_lag_17520', 'carbon_roll_24h', 'carbon_roll_168h']


In [39]:
def create_default_simulation_input(feature_cols: list) -> pd.DataFrame:
    """
    Create a one-row dataframe with sensible defaults for all required features.
    """
    default_values = {
        "temperature_2m_c": 15.0,
        "wind_speed_100m_ms": 6.0,
        "wind_gusts_10m_ms": 8.0,
        "cloud_cover_pct": 50.0,
        "shortwave_radiation_wm2": 200.0,
        "direct_radiation_wm2": 120.0,
        "diffuse_radiation_wm2": 80.0,
        "pressure_msl_hpa": 1013.0,
        "precipitation_mm": 0.0,
        "biomass": 1200.0,
        "fossil_gas": 8000.0,
        "fossil_hard_coal": 1500.0,
        "hydro_pumped_storage": 500.0,
        "hydro_run_of_river_and_poundage": 700.0,
        "nuclear": 6000.0,
        "other": 300.0,
        "solar": 2500.0,
        "wind_offshore": 4000.0,
        "wind_onshore": 3500.0,
        "totaloutput_mw": 30000.0,
        "hour_sin": 0.0,
        "hour_cos": 1.0,
        "dow_sin": 0.0,
        "dow_cos": 1.0,
        "doy_sin": 0.0,
        "doy_cos": 1.0,
        "carbon_lag_48": 180.0,
        "carbon_lag_336": 190.0,
        "carbon_lag_17520": 200.0,
        "carbon_roll_24h": 185.0,
        "carbon_roll_168h": 188.0,
    }

    row = {col: default_values.get(col, 0.0) for col in feature_cols}
    return pd.DataFrame([row], columns=feature_cols)

In [40]:
# Function to prepare prediction input
def prepare_simulation_input(df_input: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """
    Ensure the simulation dataframe matches the exact model feature order.
    """
    X = df_input.copy()
    X = X.reindex(columns=feature_cols)
    X = X.astype(float)
    return X

In [41]:
# Function to predict from a dataframe
def predict_carbon_intensity(model, df_input: pd.DataFrame, feature_cols: list) -> float:
    """
    Predict carbon intensity for a one-row input dataframe.
    """
    X = prepare_simulation_input(df_input, feature_cols)
    pred = model.predict(X)[0]
    return float(pred)

In [42]:
def apply_simulation_changes(base_df: pd.DataFrame, changes: dict) -> pd.DataFrame:
    """
    Apply user-defined changes to the baseline dataframe.

    Example:
    changes = {
        "solar": 4000,
        "wind_onshore": 5000,
        "fossil_gas": 6000
    }
    """
    sim_df = base_df.copy()

    for col, value in changes.items():
        if col not in sim_df.columns:
            raise ValueError(f"Column '{col}' not found in simulation input.")
        sim_df.at[0, col] = value

    return sim_df

In [43]:
def compare_simulation(model, base_df: pd.DataFrame, sim_df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """
    Compare baseline and simulated prediction.
    """
    baseline_pred = predict_carbon_intensity(model, base_df, feature_cols)
    simulated_pred = predict_carbon_intensity(model, sim_df, feature_cols)

    diff = simulated_pred - baseline_pred
    pct_change = (diff / baseline_pred * 100) if baseline_pred != 0 else np.nan

    result = pd.DataFrame({
        "baseline_prediction": [baseline_pred],
        "simulated_prediction": [simulated_pred],
        "absolute_change": [diff],
        "percent_change": [pct_change]
    })

    return result

In [44]:
# Create a baseline input

baseline_df = create_default_simulation_input(feature_cols)
baseline_df

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,biomass,...,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520,carbon_roll_24h,carbon_roll_168h
0,15.0,6.0,8.0,50.0,200.0,120.0,80.0,1013.0,0.0,1200.0,...,1.0,0.0,1.0,0.0,1.0,180.0,190.0,200.0,185.0,188.0


In [45]:
# Baseline prediction

baseline_prediction = predict_carbon_intensity(hgb_model, baseline_df, feature_cols)
print(f"Baseline predicted carbon intensity: {baseline_prediction:.2f}")

Baseline predicted carbon intensity: 147.50


In [46]:
# Scenario test

scenario_changes = {
    "solar": 5000.0,
    "wind_onshore": 5000.0,
    "wind_offshore": 5500.0,
    "fossil_gas": 6000.0,
    "fossil_hard_coal": 1000.0
}

scenario_df = apply_simulation_changes(baseline_df, scenario_changes)
scenario_df

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,biomass,...,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520,carbon_roll_24h,carbon_roll_168h
0,15.0,6.0,8.0,50.0,200.0,120.0,80.0,1013.0,0.0,1200.0,...,1.0,0.0,1.0,0.0,1.0,180.0,190.0,200.0,185.0,188.0


In [47]:
# Compare

comparison = compare_simulation(hgb_model, baseline_df, scenario_df, feature_cols)
comparison

,baseline_prediction,simulated_prediction,absolute_change,percent_change
0,147.499551,120.725725,-26.773826,-18.151801


In [48]:
def run_simulation(model, feature_cols: list, changes: dict, baseline_df: pd.DataFrame | None = None):
    if baseline_df is None:
        baseline_df = create_default_simulation_input(feature_cols)

    scenario_df = apply_simulation_changes(baseline_df, changes)
    comparison_df = compare_simulation(model, baseline_df, scenario_df, feature_cols)

    return baseline_df, scenario_df, comparison_df

In [49]:
changes = {
    "temperature_2m_c": 22.0,
    "cloud_cover_pct": 20.0,
    "solar": 6500.0,
    "wind_onshore": 4200.0,
    "fossil_gas": 5000.0,
    "carbon_roll_24h": 170.0,
    "carbon_roll_168h": 175.0,
}

base_df, sim_df, result_df = run_simulation(hgb_model, feature_cols, changes)

display(base_df)
display(sim_df)
display(result_df)

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,biomass,...,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520,carbon_roll_24h,carbon_roll_168h
0,15.0,6.0,8.0,50.0,200.0,120.0,80.0,1013.0,0.0,1200.0,...,1.0,0.0,1.0,0.0,1.0,180.0,190.0,200.0,185.0,188.0


,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,precipitation_mm,biomass,...,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,carbon_lag_48,carbon_lag_336,carbon_lag_17520,carbon_roll_24h,carbon_roll_168h
0,22.0,6.0,8.0,20.0,200.0,120.0,80.0,1013.0,0.0,1200.0,...,1.0,0.0,1.0,0.0,1.0,180.0,190.0,200.0,170.0,175.0


,baseline_prediction,simulated_prediction,absolute_change,percent_change
0,147.499551,116.417046,-31.082504,-21.072948
